In [1]:
# !pip install bs4 # one time run! after installed, don't worry about rerunning.
# !pip install beautifulsoup4
import requests
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from google.transit import gtfs_realtime_pb2
from google.protobuf.json_format import MessageToDict

# Loading Live Data

In [4]:
# Reads realtime GTFS from TTC API and outputs all tables list of dataframes.
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "text/plain,*/*",
}


def gtfs_to_df(url, headers):
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise an exception for HTTP errors

    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)


    entities = [
        MessageToDict(entity, preserving_proto_field_name=True)
        for entity in feed.entity
    ]
    return pd.json_normalize(entities, sep="_")


# URL of the web page we want to scrape.
feed_urls = {
    "alerts": "https://gtfsrt.ttc.ca/alerts/all?format=binary",
    "trip_updates": "https://gtfsrt.ttc.ca/trips/update?format=binary",
    "subway_trip_updates": "https://gtfsrt.ttc.ca/trips/subway?format=binary",
    "vehicle_positions": "https://gtfsrt.ttc.ca/vehicles/position?format=binary",
}

live_delays = {name: gtfs_to_df(url, headers)
              for name, url in feed_urls.items()}

live_alerts = live_delays["alerts"]
live_trip_update = live_delays["trip_updates"]
live_subway_trip_update = live_delays["subway_trip_updates"]
live_vehicle_position = live_delays["vehicle_positions"]

In [5]:
print("Alerts DataFrame:")
display(live_alerts.head())

print("Trip Updates DataFrame:")
display(live_trip_update.head())

print("Subway Trip Updates DataFrame:")
display(live_subway_trip_update.head())

print("Vehicle Positions DataFrame:")
display(live_vehicle_position.head())

Alerts DataFrame:


,id,alert_active_period,alert_informed_entity,alert_cause,alert_effect,alert_header_text_translation,alert_url_translation,alert_description_text_translation,alert_image_localized_image,alert_image_alternative_text_translation
0,79133,"[{'start': '1793538000', 'end': '1793552400'}]","[{'route_id': '2', 'stop_id': '13777'}, {'rout...",MAINTENANCE,NO_SERVICE,[{'text': 'Line 2 Bloor-Danforth: There is no ...,NaN,NaN,NaN,NaN
1,79132,"[{'start': '1793216941', 'end': '1793538000'}]","[{'route_id': '2', 'stop_id': '13777'}, {'rout...",MAINTENANCE,REDUCED_SERVICE,"[{'text': 'Line 2 Bloor-Danforth: On Sunday, N...",NaN,NaN,NaN,NaN
2,79129,"[{'start': '1793412000', 'end': '1793431800'}]","[{'route_id': '2', 'stop_id': '13856'}, {'rout...",MAINTENANCE,NO_SERVICE,[{'text': 'Line 2 Bloor-Danforth: There is no ...,NaN,NaN,NaN,NaN
3,79128,"[{'start': '1793215415', 'end': '1793412000'}]","[{'route_id': '2', 'stop_id': '13856'}, {'rout...",MAINTENANCE,REDUCED_SERVICE,[{'text': 'Line 2 Bloor-Danforth: There will b...,NaN,NaN,NaN,NaN
4,78762,"[{'start': '1793332740', 'end': '1793345400'}]","[{'route_id': '1', 'stop_id': '13836'}, {'rout...",MAINTENANCE,NO_SERVICE,[{'text': 'Line 1 Yonge-University: There is n...,NaN,NaN,NaN,NaN


Trip Updates DataFrame:


,id,trip_update_trip_trip_id,trip_update_trip_schedule_relationship,trip_update_trip_route_id,trip_update_stop_time_update,trip_update_vehicle_id,trip_update_timestamp
0,1,-1469397216,DELETED,100,"[{'stop_sequence': 1, 'departure': {'time': '1...",6602,1793221060
1,2,-214174612,DELETED,600,"[{'stop_sequence': 1, 'departure': {'time': '1...",NaN,1793221064
2,3,-1215508714,DELETED,65,"[{'stop_sequence': 1, 'departure': {'time': '1...",NaN,1793221064
3,4,-173211500,DELETED,600,"[{'stop_sequence': 1, 'departure': {'time': '1...",NaN,1793221064
4,5,-1842380438,DELETED,996,"[{'stop_sequence': 1, 'departure': {'time': '1...",NaN,1793221064


Subway Trip Updates DataFrame:


,id,trip_update_trip_trip_id,trip_update_trip_schedule_relationship,trip_update_trip_route_id,trip_update_stop_time_update,trip_update_vehicle_id,trip_update_vehicle_label
0,subway-134928775|North,134928775,SCHEDULED,1,"[{'stop_sequence': 5, 'arrival': {'time': '179...",43,43
1,subway-134928747|North,134928747,SCHEDULED,1,"[{'stop_sequence': 6, 'arrival': {'time': '179...",147,147
2,subway-134928376|South,134928376,SCHEDULED,1,"[{'stop_sequence': 12, 'arrival': {'time': '17...",77,77
3,subway-134929550|West,134929550,SCHEDULED,2,"[{'stop_sequence': 8, 'arrival': {'time': '179...",47,47
4,subway-134928784|South,134928784,SCHEDULED,1,"[{'stop_sequence': 1, 'arrival': {'time': '179...",52,52


Vehicle Positions DataFrame:


,id,vehicle_trip_trip_id,vehicle_trip_schedule_relationship,vehicle_trip_route_id,vehicle_position_latitude,vehicle_position_longitude,vehicle_position_bearing,vehicle_position_speed,vehicle_current_stop_sequence,vehicle_current_status,vehicle_timestamp,vehicle_stop_id,vehicle_vehicle_id,vehicle_occupancy_status
0,1,72061020,SCHEDULED,84,43.733917,-79.537900,253.0,0.0000,39.0,INCOMING_AT,1793221028,6526,3640,EMPTY
1,2,100750020,SCHEDULED,906,43.686770,-79.593925,87.0,13.4112,5.0,IN_TRANSIT_TO,1793221035,9578,3638,EMPTY
2,3,40815020,SCHEDULED,952,43.690945,-79.573300,74.0,0.0000,5.0,INCOMING_AT,1793221034,9729,3639,FEW_SEATS_AVAILABLE
3,4,68078020,SCHEDULED,906,43.725952,-79.622055,344.0,0.0000,3.0,IN_TRANSIT_TO,1793221021,7264,3630,EMPTY
4,5,46129020,SCHEDULED,996,43.719875,-79.513250,73.0,0.0000,22.0,IN_TRANSIT_TO,1793221032,8551,3631,FEW_SEATS_AVAILABLE


# Loading Historical Data

Loads bus delays historical .csv data from the TTC website.

In [42]:
bus_delays = {}

# Toronto Open Data is stored in a CKAN instance. It's APIs are documented here:
# https://docs.ckan.org/en/latest/api/


# To hit our API, you'll be making requests to:
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"


# Datasets are called "packages". Each package can contain many "resources"
# To retrieve the metadata for this package and its resources, use the package name in this page's URL:

url = base_url + "/api/3/action/package_show"
params = {"id": "ttc-bus-delay-data"}
package = requests.get(url, params=params).json()

# To get resource data:
for idx, resource in enumerate(package["result"]["resources"]):
    # To get metadata for non datastore_active resources:

    if not resource["datastore_active"]:
        url = base_url + "/api/3/action/resource_show?id=" + resource["id"]
        resource_metadata = requests.get(url).json()

        url = resource_metadata["result"]["url"]
        name = resource_metadata["result"]["name"].replace(".csv", "").replace(".xlsx", "")

        # From here, you can use the "url" attribute to download this file
        # This only donwloads the .csv and.xlsx files and makes sure not to download multiple copies of the same package.
        if url.endswith(".csv"):
            bus_delays[name] = pd.read_csv(url)
        elif url.endswith(".xlsx"):
            bus_delays[name] = pd.read_excel(url)




## Loading GTFS system data

Loads GTFS data from TTC website (info about routes, stops, polygons, etc.)

In [43]:
# Reads in TTC information for GTFS (Stop times, routes, trips, etc.) from Toronto Open Data and outputs all tables as dictionary of dataframes.
import zipfile
import io

gtfs_merged = {}

url = base_url + "/api/3/action/package_show"

params = {"id": "merged-gtfs-ttc-routes-and-schedules"}

package = requests.get(url, params=params).json()

# To get resource data:

for idx, resource in enumerate(package["result"]["resources"]):

    # To get metadata for non datastore_active resources:

    if not resource["datastore_active"]:

        url = base_url + "/api/3/action/resource_show?id=" + resource["id"]

        resource_metadata = requests.get(url).json()

        print(resource_metadata)

        # From here, you can use the "url" attribute to download this file
        url = resource_metadata["result"]["url"]

        # Reads each file in the zip and stores it in a dictionary of dataframes.
        with zipfile.ZipFile(io.BytesIO(requests.get(url).content)) as z:
            for file in z.namelist():
                    with z.open(file) as f:
                        gtfs_merged[file] = pd.read_csv(f)


{'help': 'https://ckan0.cf.opendata.inter.prod-toronto.ca/api/3/action/help_show?name=resource_show', 'success': True, 'result': {'cache_last_updated': None, 'cache_url': None, 'created': '2025-10-14T16:02:48.643575', 'datastore_active': False, 'datastore_cache': [''], 'datastore_resource_id': '', 'format': 'ZIP', 'hash': '', 'id': 'c920e221-7a1c-488b-8c5b-6d8cd4e85eaf', 'is_datastore_cache_file': False, 'last_modified': '2026-09-30T14:17:52', 'metadata_modified': '2026-09-30T15:14:23.916048', 'mimetype': 'application/zip', 'mimetype_inner': None, 'name': 'Complete GTFS', 'package_id': 'b811ead4-6eaf-4adb-8408-d389fb5a069c', 'position': 0, 'resource_type': None, 'size': 84337545, 'state': 'active', 'url': 'https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/b811ead4-6eaf-4adb-8408-d389fb5a069c/resource/c920e221-7a1c-488b-8c5b-6d8cd4e85eaf/download/completegtfs.zip', 'url_type': 'upload'}}


C:\Users\nikit\AppData\Local\Temp\ipykernel_7268\200269924.py:34: DtypeWarning: Columns (4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  gtfs_merged[file] = pd.read_csv(f)
C:\Users\nikit\AppData\Local\Temp\ipykernel_7268\200269924.py:34: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  gtfs_merged[file] = pd.read_csv(f)


The following keys are the files that correspnd with the GTFS feed

In [44]:
gtfs_merged.keys()

dict_keys(['stops.txt', 'trips.txt', 'agency.txt', 'calendar.txt', 'calendar_dates.txt', 'feed_info.txt', 'levels.txt', 'pathways.txt', 'routes.txt', 'shapes.txt', 'stop_times.txt'])

In [45]:
bus_codes = bus_delays["Code Descriptions"]
# Cleans up and symbols
bus_codes["DESCRIPTION"] = bus_codes["DESCRIPTION"].str.replace("â\x80\x93", "&")

# combine all the dataframes into one dataframe, ignoring the index and keeping only the dataframes that have yearly data.
bus_delays_data= pd.concat({name: df for name, df in bus_delays.items() if "data" in name.lower() and "readme" not in name},ignore_index=True)

# convert Code to description and consolidate the code and incide columens
bus_delays_data = bus_delays_data.merge(bus_codes, how="left", left_on="Code", right_on="CODE")

#fix mismatching column names throughout the years to match 2025 and beyond data.
bus_delays_data["Date"] = bus_delays_data["Date"].fillna(bus_delays_data["Report Date"])
bus_delays_data["Min Delay"] = bus_delays_data["Min Delay"].fillna(bus_delays_data["Delay"])
bus_delays_data["Min Gap"] = bus_delays_data["Min Gap"].fillna(bus_delays_data["Gap"])
bus_delays_data["Line"] = bus_delays_data["Line"].fillna(bus_delays_data["Route"])
bus_delays_data["Station"] = bus_delays_data["Station"].fillna(bus_delays_data["Location"])
bus_delays_data["Incident"] = bus_delays_data["Incident"].fillna(bus_delays_data["DESCRIPTION"])
    
bus_delays_data["Date"] = pd.to_datetime(bus_delays_data["Date"], errors="coerce")

bus_delays_data.drop(columns=["Report Date", "Delay", "Gap", "Location", "DESCRIPTION"], inplace=True)


#reorder columns to match the order of the columns in the 2025 data. Incident added, will edit to get relavent code name.
bus_delays_data = bus_delays_data[["Date", "Line", "Time", "Day","Station", "Code","Incident", "Min Delay", "Min Gap", "Bound", "Vehicle"]]

display(bus_delays_data.head())
display(bus_delays_data.tail())

,Date,Line,Time,Day,Station,Code,Incident,Min Delay,Min Gap,Bound,Vehicle
0,2014-01-01,95,00:23:00,Wednesday,York Mills station,NaN,Mechanical,10.0,20.0,NaN,1734.0
1,2014-01-01,102,00:55:00,Wednesday,Entire run for route,NaN,General Delay,33.0,66.0,NaN,8110.0
2,2014-01-01,54,01:28:00,Wednesday,lawrence and Warden,NaN,Mechanical,10.0,20.0,NaN,7478.0
3,2014-01-01,112,01:30:00,Wednesday,Kipling Station,NaN,Emergency Services,18.0,36.0,NaN,8084.0
4,2014-01-01,24,01:37:00,Wednesday,VP and Ellesmere,NaN,Investigation,10.0,20.0,NaN,7843.0


,Date,Line,Time,Day,Station,Code,Incident,Min Delay,Min Gap,Bound,Vehicle
329840,2026-09-30,21 BRIMLEY,00:07,Wednesday,BRIMLEY AND STEELES,MFDV,ON DIVERSION,37.0,57.0,N,8599.0
329841,2026-09-30,149 ETOBICOKE-BLOOR,00:33,Wednesday,BLOOR AND OLD MILL TRA,EFO,OTHER,20.0,40.0,E,3361.0
329842,2026-09-30,94 WELLESLEY,00:44,Wednesday,WELLESLEY STATION,SFDP,DISORDERLY PATRON,11.0,22.0,NaN,8469.0
329843,2026-09-30,65 PARLIAMENT,01:31,Wednesday,QUEENS QUAY AND SHERBO,MFBIO,NaN,13.0,26.0,N,8513.0
329844,2026-09-30,54 LAWRENCE EAST,01:40,Wednesday,LAWRENCE AVENUE EAST A,SFL,NaN,0.0,0.0,W,3147.0


Some of the older records don't have codes or incidents related to weather before 2025, these will be handled seperately by Michelle. I'll segement out the delays that do have codes related to weather.

In [46]:
weather_bus_delays=bus_delays_data[bus_delays_data["Incident"].str.contains("weather", case=False, na=False)]
weather_bus_delays.head()

,Date,Line,Time,Day,Station,Code,Incident,Min Delay,Min Gap,Bound,Vehicle
223661,2025-01-10,929 DUFFERIN EXPRESS,21:47,Friday,DUFFERIN AND ROGERS,MFWEA,WEATHER RELATED,0.0,0.0,N,9146.0
223681,2025-01-10,7 BATHURST,00:52,Friday,BATHURST AND DAVENPORT,MFWEA,WEATHER RELATED,10.0,20.0,N,9021.0
223689,2025-01-11,320 YONGE,02:15,Saturday,YONGE AND WELLESLEY,MFWEA,WEATHER RELATED,0.0,0.0,NaN,0.0
223694,2025-01-11,307 BATHURST,03:10,Saturday,BATHURST AND CEDARCROF,MFWEA,WEATHER RELATED,0.0,0.0,NaN,0.0
223700,2025-01-11,903 KENNEDY SCARBOROUG,06:00,Saturday,CENTENNIAL COLLEGE,MFWEA,WEATHER RELATED,0.0,0.0,S,9403.0


# Including Geographic Data to Delays

## Problem Statement

There is an issue with the data sets that the delay station names =/= stop names. Aditionally there are multiple stops with the same stop names (i.e. if there are 4 stops at the 4 legs of an intersection). 

Current proposed methodology: 
- For each delay, use regex (or AI implementation later on) to find stops
    - Reject rejex for now due to strings being too different in some occurances.
- Take the average lat and long to get an average stop location for the station.
- Once modeling is done on a stop basis, include route data as well.

In [47]:
display(weather_bus_delays.head(1))
display(gtfs_merged["stops.txt"][gtfs_merged["stops.txt"]["stop_name"].str.upper().str.contains("(?=.*DUFFERIN)(?=.*ROGERS)", regex=True)])

,Date,Line,Time,Day,Station,Code,Incident,Min Delay,Min Gap,Bound,Vehicle
223661,2025-01-10,929 DUFFERIN EXPRESS,21:47,Friday,DUFFERIN AND ROGERS,MFWEA,WEATHER RELATED,0.0,0.0,N,9146.0


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,level_id,tts_stop_name
1461,2082,2082,Dufferin St at Rogers Rd,NaN,43.685461,-79.445973,NaN,NaN,0,NaN,NaN,1,NaN,NaN
1462,2083,2083,Dufferin St at Rogers Rd,NaN,43.685673,-79.446257,NaN,NaN,0,NaN,NaN,1,NaN,NaN
4121,6020,6020,Rogers Rd at Dufferin St,NaN,43.685463,-79.446245,NaN,NaN,0,NaN,NaN,1,NaN,NaN
4122,6021,6021,Rogers Rd at Dufferin St,NaN,43.685645,-79.445975,NaN,NaN,0,NaN,NaN,1,NaN,NaN


## 1. Stop Aggregation and Merge Current Matches

This section takes all the unique station names in the delay datafram and matches it with a stop name and location provided in the GTFS stops.txt file

In [48]:
import string
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
english_stopwords = set(stopwords.words('english'))


def clean_station_stop_name(name, replace = True):
    if pd.isna(name):
        return ""

    name = name.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
    name = name.lower()  # convert to lowercase
    if replace: 
        remove_words = [
        # remove common TTC words
            " station",
            " stn",
            " bus terminal",
            " terminal",
            " loop",
            " subway",
            " st",            
            " street",    
            " southbound",
            " northbound",
            " eastbound",
            " westbound",  
            " Road",
            " road",
            " rd"
        ]

        for word in remove_words:
            name = name.replace(word, "")
    
    stations = str(name).split()

    stations_filtered = " ".join([ station for station in stations if station not in english_stopwords])

    return stations_filtered

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nikit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Filter and clean stop names to remove common stopping words. Addtiotionally, get unique stations in the bus delays 

In [49]:
ttc_stops = gtfs_merged["stops.txt"]
ttc_stops = ttc_stops.filter(items=["stop_name", "stop_lat", "stop_lon"]).dropna()
ttc_stops["stop_name"] = ttc_stops["stop_name"].str.lower()
ttc_stops["stop_name"] = ttc_stops["stop_name"].apply(clean_station_stop_name)


# Group by exact stop name and average locations.Results in unique stop names.
ttc_stops = ttc_stops.groupby("stop_name").mean()

# Get unique station names, drop everything else from df. 847 of them
station_names = pd.DataFrame(weather_bus_delays["Station"].dropna().unique(), columns=["Station"])
station_names

,Station
0,DUFFERIN AND ROGERS
1,BATHURST AND DAVENPORT
2,YONGE AND WELLESLEY
3,BATHURST AND CEDARCROF
4,CENTENNIAL COLLEGE
...,...
866,WEST ROAD AND COLBORNE
867,SHEPPARD AND BAKERS FI
868,SOUTH AND GLEN
869,SOUTH DRIVE AND GLEN


## 2. Normalize Station Names Data

This section parses out common intersection crossings and matches the stop/station name with stops.txt

In [50]:
split_station_names = station_names[station_names["Station"].str.contains(" AND ")]["Station"].str.split(" AND ")
display(split_station_names)
split_station_names = [ttc_stops[ttc_stops.index.str.contains(station[0].strip(), case=False, regex=False) & ttc_stops.index.str.contains(station[1].strip(), case=False, regex=False)].assign(station_idx=station_idx) for station_idx, station in split_station_names.items()]

0         [DUFFERIN, ROGERS]
1      [BATHURST, DAVENPORT]
2         [YONGE, WELLESLEY]
3      [BATHURST, CEDARCROF]
5      [AVENUE, ST CLAIR WE]
               ...          
866    [WEST ROAD, COLBORNE]
867    [SHEPPARD, BAKERS FI]
868            [SOUTH, GLEN]
869      [SOUTH DRIVE, GLEN]
870    [CORBURN LODGE, BLOO]
Name: Station, Length: 713, dtype: object

Group all station that contain the intersection streets and get the average latitude and longitude values if the stops are within resonable distance of each other (~20m)

In [51]:
# takes non empty matches and averages the lat and lon that are close together, but it seems to work well for the data. Will merge with geopandas later
split_station_names = [group for group in split_station_names if group is not None and not group.empty]

station_names_merged = [{group["station_idx"].iloc[0]: [group["stop_lat"].mean(), group["stop_lon"].mean()]} for group in split_station_names if [
    groups["stop_lat"].max()-groups["stop_lat"].min() < 0.0002 for groups in split_station_names]]

station_names_merged = pd.DataFrame({k: v for d in station_names_merged for k, v in d.items(
)}).T.rename(columns={0: "stop_lat", 1: "stop_lon"})

Fill in the matched stations to the original df of unique station names. First do an exact match then match the cleaned up station names.

In [52]:
station_names = station_names.merge(
    station_names_merged, left_index=True, right_index=True, how="left")

print("Not null:", len(
    station_names[~station_names["stop_lat"].isnull()]))
print("Null:", len(
    station_names[station_names["stop_lat"].isnull()]))

Not null: 560
Null: 311


In [53]:
# merge cleaned up station names with merged ttc stops
station_names = station_names.merge(
    ttc_stops, left_on=station_names["Station"].apply(clean_station_stop_name), 
    right_index=True, how="left", suffixes=("", "_merged")
    )
station_names= station_names.fillna({"stop_lat": station_names["stop_lat_merged"], "stop_lon": station_names["stop_lon_merged"]}).drop(columns=["stop_lat_merged", "stop_lon_merged"])
print("Not null:", len(
    station_names[~station_names["stop_lat"].isnull()]))
print("Null:", len(
    station_names[station_names["stop_lat"].isnull()]))

Not null: 614
Null: 257


## 3. LLM Implementation

There are still many unmatched stations due to: non existent streets, mismatching names,invalid naming/location etc. This section takes the rest of the unmatched stations and runs it through an LLM which cross references TTC website and general google maps to get latitude and longitude of these stations.

Load in chat gpt generated csv matching the stop names

In [54]:
gpt_station_names = pd.read_csv("chatgpt_station_names.csv", index_col=0)
# merge cleaned up station names with data frame of unique station names
station_names = station_names.merge(gpt_station_names, left_index=True, right_index=True, how="left", suffixes=("", "_gpt"))
station_names= station_names.fillna({"stop_lat": station_names["stop_lat_gpt"], "stop_lon": station_names["stop_lon_gpt"]}).drop(columns=["stop_lat_gpt", "stop_lon_gpt","Station_gpt"])
display(station_names)

,Station,stop_lat,stop_lon
0,DUFFERIN AND ROGERS,43.685560,-79.446112
1,BATHURST AND DAVENPORT,43.676738,-79.415920
2,YONGE AND WELLESLEY,43.664919,-79.384585
3,BATHURST AND CEDARCROF,43.783511,-79.445589
4,CENTENNIAL COLLEGE,43.786177,-79.229530
...,...,...,...
866,WEST ROAD AND COLBORNE,43.653432,-79.465379
867,SHEPPARD AND BAKERS FI,43.754319,-79.480593
868,SOUTH AND GLEN,43.706334,-79.390488
869,SOUTH DRIVE AND GLEN,43.677632,-79.375341


All remaining na lat/lon values are either parallel streets, routes, or general areas or entire routes. There are 39 of them

In [55]:
station_names[station_names["stop_lat"].isnull()]

,Station,stop_lat,stop_lon
13,ST PHILLIPS AND MARTIN,NaN,NaN
14,ST PHILLIPS - MARTIN G,NaN,NaN
63,MORNINGSIDE AND DURIE,NaN,NaN
84,ENTIRE ROUTE,NaN,NaN
123,SHEPPARD AVE WEST AND,NaN,NaN
132,DAVISVILLE AND THE BRI,NaN,NaN
228,MILLWOOD OVERLEA TO DO,NaN,NaN
251,25 DON MILLS ROUTE,NaN,NaN
252,939 FINCH EXPRESS ROUT,NaN,NaN
253,72 PAPE ROUTE,NaN,NaN


Merge the delay data to the unique station names

In [56]:
weather_bus_delays = weather_bus_delays.merge(station_names, how="left", left_on="Station", right_on="Station")
display(weather_bus_delays)

,Date,Line,Time,Day,Station,Code,Incident,Min Delay,Min Gap,Bound,Vehicle,stop_lat,stop_lon
0,2025-01-10,929 DUFFERIN EXPRESS,21:47,Friday,DUFFERIN AND ROGERS,MFWEA,WEATHER RELATED,0.0,0.0,N,9146.0,43.685560,-79.446112
1,2025-01-10,7 BATHURST,00:52,Friday,BATHURST AND DAVENPORT,MFWEA,WEATHER RELATED,10.0,20.0,N,9021.0,43.676738,-79.415920
2,2025-01-11,320 YONGE,02:15,Saturday,YONGE AND WELLESLEY,MFWEA,WEATHER RELATED,0.0,0.0,NaN,0.0,43.664919,-79.384585
3,2025-01-11,307 BATHURST,03:10,Saturday,BATHURST AND CEDARCROF,MFWEA,WEATHER RELATED,0.0,0.0,NaN,0.0,43.783511,-79.445589
4,2025-01-11,903 KENNEDY SCARBOROUG,06:00,Saturday,CENTENNIAL COLLEGE,MFWEA,WEATHER RELATED,0.0,0.0,S,9403.0,43.786177,-79.229530
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1328,2026-09-03,101 DOWNSVIEW PARK,08:51,Thursday,SHEPPARD AND CHESSWOOD,MFWEA,WEATHER RELATED,420.0,420.0,NaN,7075.0,43.755540,-79.474377
1329,2026-09-03,82 ROSEDALE,15:05,Thursday,SOUTH AND GLEN,MFWEA,WEATHER RELATED,108.0,110.0,NaN,8904.0,43.706334,-79.390488
1330,2026-09-04,75 SHERBOURNE,06:37,Friday,SOUTH DRIVE AND GLEN,MFWEA,WEATHER RELATED,672.0,684.0,S,8465.0,43.677632,-79.375341
1331,2026-09-04,82 ROSEDALE,06:42,Friday,SOUTH DRIVE AND GLEN,MFWEA,WEATHER RELATED,675.0,692.0,NaN,8537.0,43.677632,-79.375341


Remianing null values. Assuming the remaining values indicate entire route is delayed or portion of route delayed. How do we want to deal with entire route? Will be ignored for now but need to be incorporated

In [57]:
weather_bus_delays[weather_bus_delays["stop_lat"].isnull()]

,Date,Line,Time,Day,Station,Code,Incident,Min Delay,Min Gap,Bound,Vehicle,stop_lat,stop_lon
13,2025-01-28,52 LAWRENCE WEST,18:48,Tuesday,ST PHILLIPS AND MARTIN,MFWEA,WEATHER RELATED,0.0,0.0,E,8758.0,NaN,NaN
14,2025-01-28,952 LAWRENCE WEST EXPR,18:49,Tuesday,ST PHILLIPS - MARTIN G,MFWEA,WEATHER RELATED,0.0,0.0,E,3605.0,NaN,NaN
68,2025-02-08,77 SWANSEA,21:30,Saturday,MORNINGSIDE AND DURIE,MFWEA,WEATHER RELATED,265.0,275.0,W,7217.0,NaN,NaN
95,2025-02-09,86 SCARBOROUGH,07:48,Sunday,ENTIRE ROUTE,MFWEA,WEATHER RELATED,20.0,20.0,NaN,0.0,NaN,NaN
100,2025-02-12,77 SWANSEA,22:48,Wednesday,MORNINGSIDE AND DURIE,MFWEA,WEATHER RELATED,0.0,0.0,NaN,7211.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1170,2026-01-29,924 VICTORIA PARK EXPR,15:44,Thursday,ENTIRE ROUTE,MFWEA,WEATHER RELATED,0.0,0.0,NaN,0.0,NaN,NaN
1200,2026-02-13,39 FINCH EAST,16:00,Friday,ENTIRE ROUTE,MFWEA,WEATHER RELATED,0.0,0.0,NaN,0.0,NaN,NaN
1209,2026-02-18,129 MCCOWAN NORTH,11:29,Wednesday,KENNEDY AND RANCHOR,MFWEA,WEATHER RELATED,0.0,0.0,N,3243.0,NaN,NaN
1235,2026-02-18,116 MORNINGSIDE,16:20,Wednesday,39 GUILDPARK PATHWAY,MFWEA,WEATHER RELATED,0.0,0.0,W,3472.0,NaN,NaN


# Rain Guage Locations

In [2]:
import geopandas as gpd
from pathlib import Path
from html import escape

import folium
import geopandas as gpd
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from branca.colormap import linear
from folium.features import DivIcon, GeoJsonTooltip
from IPython.display import display, HTML
from scipy.spatial import Voronoi
from shapely.geometry import Polygon
import os

%matplotlib inline


## 1. Load Data for TTC API

Will only load 2025-2026 data due to the fact delays from 2025-2026 will be used

In [3]:
precipitation_data_frames ={}

import requests

base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"

url = base_url + "/api/3/action/package_show"
params = {"id": "rain-gauge-locations-and-precipitation"}
package = requests.get(url, params=params).json()

for idx, resource in enumerate(package["result"]["resources"]):

    if not resource["datastore_active"]:
        url = base_url + "/api/3/action/resource_show?id=" + resource["id"]
        resource_metadata = requests.get(url).json()
        print(resource_metadata)

        url = resource_metadata["result"]["url"]
        name = resource_metadata["result"]["name"].replace(".csv", "").replace(".xlsx", "")

        # From here, you can use the "url" attribute to download this file
        # This only donwloads the .csv and.xlsx files and makes sure not to download multiple copies of the same package.
        if url.endswith(".csv"):
            precipitation_data_frames[name] = pd.read_csv(url)
        # From here, you can use the "url" attribute to download this file

{'help': 'https://ckan0.cf.opendata.inter.prod-toronto.ca/api/3/action/help_show?name=resource_show', 'success': True, 'result': {'cache_last_updated': None, 'cache_url': None, 'created': '2020-12-07T21:55:06.111448', 'datastore_active': False, 'extract_job': 'Airflow: upload_remote_files', 'format': 'ZIP', 'hash': '', 'id': 'e93a7088-d37b-484e-addb-0f9eec4dfa19', 'is_datastore_cache_file': False, 'is_preview': False, 'last_modified': '2020-04-15T06:00:03', 'metadata_modified': '2022-03-10T20:40:57.206456', 'mimetype': 'application/zip', 'mimetype_inner': None, 'name': 'Rain gauge notes', 'package_id': 'f2933501-0373-4734-b50c-4e4f39646180', 'position': 0, 'resource_type': None, 'revision_id': '8a3f263b-2de6-4c84-ad7a-c1406a11bc27', 'size': 25160, 'state': 'active', 'url': 'https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/f2933501-0373-4734-b50c-4e4f39646180/resource/e93a7088-d37b-484e-addb-0f9eec4dfa19/download/raingaugenotes.zip', 'url_type': 'upload'}}
{'help': 'https://ckan0

## 2. Cleanup Gauge and bus delay data and Join

Make bus delay data a geopandas dataframe

In [83]:
weather_bus_delays_gdf = gpd.GeoDataFrame(
    weather_bus_delays,
    geometry=gpd.points_from_xy(
        weather_bus_delays["stop_lon"], weather_bus_delays["stop_lat"]),
    crs="EPSG:4326"
)

In [16]:
# Get precipitation data from 2025 and 2026 and merge data frames into one data frame. This will be used to analyze the relationship between bus delays and precipitation.
for year in [2025, 2026]:
    precipitation_data = precipitation_data_frames.get(f"precipitation-data-{year}")
    if precipitation_data is not None:
        precipitation_data["date"] = pd.to_datetime(precipitation_data["date"], errors="coerce")
        precipitation_data["rainfall"] = pd.to_numeric(precipitation_data["rainfall"], errors="coerce")
        if "precipitation" in locals():
            precipitation = pd.concat([precipitation, precipitation_data], ignore_index=True)
        else:
            precipitation = precipitation_data.copy()

In [80]:
# Cleanup precipitation and segment data fram based on season

# Remove rows with missing or invalid values in essential columns
precipitation = precipitation.dropna(
    subset=["date", "name", "longitude", "latitude", "rainfall"]
).copy()

# Remove rows with invalid rainfall values
precipitation = precipitation[precipitation["rainfall"] >= 0]

# Remove exact duplicate rows
precipitation = precipitation.drop_duplicates()

# Aggregate timestamps to calendar days so all observations from the same day
# are included in the daily gauge total.
precipitation["date"] = precipitation["date"].dt.normalize()


# Creating a dataframe of unique rain gauges. Drop duplicates to get unique gauges, sort by name in ascending order, and set the index as the name.
gauges = (
    precipitation[["name", "longitude", "latitude"]]
    .drop_duplicates(subset="name")
    .sort_values("name")
    .reset_index(drop=True)
)

# Indexes the dataframe by the date.
rainfall_by_time = precipitation.pivot_table(
    index="date",
    columns="name",
    values="rainfall",
    aggfunc="sum",
).reindex(columns=gauges["name"])

# Only guages on from Nov 1st to Mar 31st
four_season_gauges = gauges[gauges["name"].isin(
    ["RG_016", "RG_017", "RG_018", "RG_019", "RG_038"])].reset_index(drop=True)

# Sum all rainfall observations for each gauge on each calendar day.
daily_precipitation = precipitation.copy()
daily_precipitation["date"] = daily_precipitation["date"].dt.normalize()
daily_precipitation = (
    precipitation.groupby(["date", "name"], as_index=False, observed=True)["rainfall"]
    .sum()
)

all_precipitation = daily_precipitation[(
    daily_precipitation["date"].dt.month >= 4) & (daily_precipitation["date"].dt.month <= 10)]
winter_precipitation = daily_precipitation[~daily_precipitation['date'].isin(all_precipitation['date'])]

## 2. Create Thessian Polygons

Define function for making thessian polygons

In [ ]:
def finite_voronoi_regions(voronoi_diagram, radius):
    regions = []
    vertices = voronoi_diagram.vertices.tolist()
    center = voronoi_diagram.points.mean(axis=0)
    ridge_map = {}
    for (point_a, point_b), (vertex_a, vertex_b) in zip(voronoi_diagram.ridge_points, voronoi_diagram.ridge_vertices):
        ridge_map.setdefault(point_a, []).append((point_b, vertex_a, vertex_b))
        ridge_map.setdefault(point_b, []).append((point_a, vertex_a, vertex_b))

    for point_index, region_index in enumerate(voronoi_diagram.point_region):
        region = voronoi_diagram.regions[region_index]
        if all(vertex >= 0 for vertex in region):
            regions.append(region)
            continue

        finite_vertices = [vertex for vertex in region if vertex >= 0]
        for other_point, vertex_a, vertex_b in ridge_map[point_index]:
            if vertex_a >= 0 and vertex_b >= 0:
                continue
            finite_vertex = vertex_a if vertex_a >= 0 else vertex_b
            tangent = voronoi_diagram.points[other_point] - \
                voronoi_diagram.points[point_index]
            tangent /= np.linalg.norm(tangent)
            normal = np.array([-tangent[1], tangent[0]])
            midpoint = voronoi_diagram.points[[
                point_index, other_point]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, normal)) * normal
            vertices.append(
                (voronoi_diagram.vertices[finite_vertex] + direction * radius).tolist())
            finite_vertices.append(len(vertices) - 1)

        region_coordinates = np.asarray(
            [vertices[index] for index in finite_vertices])
        region_center = region_coordinates.mean(axis=0)
        order = np.argsort(np.arctan2(
            region_coordinates[:, 1] - region_center[1], region_coordinates[:, 0] - region_center[0]))
        regions.append(np.asarray(finite_vertices)[order].tolist())
    return regions, np.asarray(vertices)

#Will need to use an acutal polygon for GTA here rather than just the convex hull of the gauges. This will ensure that the Thiessen polygons are clipped to the actual study area rather than just the convex hull of the gauges.
def make_thiessen_polygons(gauges):
    projected_crs = "EPSG:4326"  # WGS 84 / UTM zone 17N
    gauges_gdf = gpd.GeoDataFrame(
        gauges,
        geometry=gpd.points_from_xy(gauges["longitude"], gauges["latitude"]),
        crs="EPSG:4326",
    ).to_crs(projected_crs)
    coordinates = np.column_stack([gauges_gdf.geometry.x, gauges_gdf.geometry.y])
    if len(np.unique(coordinates, axis=0)) != len(coordinates):
        raise ValueError(
            "Two or more gauges have identical coordinates; Thiessen regions cannot be assigned unambiguously.")
    if np.linalg.matrix_rank(coordinates - coordinates.mean(axis=0)) < 2:
        raise ValueError(
            "Gauge locations are collinear; a two-dimensional Thiessen map cannot be created.")

    voronoi = Voronoi(coordinates)

    radius = np.ptp(coordinates, axis=0).max() * 4
    regions, voronoi_vertices = finite_voronoi_regions(voronoi, radius)
    study_area = gauges_gdf.geometry.union_all().convex_hull
    polygons = [Polygon(voronoi_vertices[region]).intersection(
        study_area) for region in regions]
    if any(polygon.is_empty or not polygon.is_valid for polygon in polygons):
        raise ValueError(
            "A Thiessen polygon is empty or invalid after clipping to the gauge convex hull.")

    thiessen = gpd.GeoDataFrame(
        {"gauge": gauges_gdf["name"].to_list()},
        geometry=polygons,
        crs=projected_crs,
    )
    return thiessen


thiessen = make_thiessen_polygons(gauges)
winter_thiessen = make_thiessen_polygons(four_season_gauges)

In [ ]:
folium_map = folium.Map(
    location=[43.7, -79.4],
    zoom_start=10,
    tiles="OpenStreetMap",
    attr="© OpenStreetMap contributors",
)
rain_gauges_group = folium.FeatureGroup(name="Rain Gauges", show=True)
rain_gauges_group.add_to(folium_map)
folium.map.CustomPane("polygons", z_index=400).add_to(folium_map)
folium.map.CustomPane("markers", z_index=650).add_to(folium_map)

winter_rain_gauges_group = folium.FeatureGroup(
    name="Winter Rain Gauges", show=True)
winter_rain_gauges_group.add_to(folium_map)

folium.GeoJson(
    thiessen,
    name="Thiessen Polygons",
    style_function=lambda x: {
        "fillColor": "darkblue",
        "color": "darkblue",
        "weight": 1,
        "fillOpacity": 0.5,
    },
).add_to(rain_gauges_group)

folium.GeoJson(
    winter_thiessen,
    name="Winter Thiessen Polygons",
    style_function=lambda x: {
        "fillColor": "blue",
        "color": "blue",
        "weight": 1,
        "fillOpacity": 0.5,
    },
).add_to(winter_rain_gauges_group)


for _, row in gauges.iterrows():
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=folium.Popup(
            f"<b>Gauge Name:</b> {escape(row['name'])}<br>"
            f"<b>Latitude:</b> {row['latitude']}<br>"
            f"<b>Longitude:</b> {row['longitude']}",
            max_width=300,
        ),
        icon=folium.Icon(color="red", icon="tint", prefix="fa"),
    ).add_to(rain_gauges_group)



folium.GeoJson(
    gpd.GeoDataFrame(four_season_gauges, geometry=gpd.points_from_xy(
        four_season_gauges["longitude"], four_season_gauges["latitude"]), crs="EPSG:4326"),
    name="Winter Rain Gauges",
    style_function=lambda x: {
        "fillColor": "darkblue",
        "color": "darkblue",
        "weight": 1,
        "fillOpacity": 0.5,
    },
).add_to(winter_rain_gauges_group)

folium.LayerControl().add_to(folium_map)

folium_map.save("thiessen_polygons_map.html")
folium_map

In [ ]:
all_precipitation = all_precipitation.sort_values("rainfall", ascending=False).set_index("date")
display(all_precipitation)

,name,rainfall
date,,
2025-07-24,RG_059,132.579000
2026-09-30,RG_052,106.439000
2026-09-02,RG_027,106.007000
2026-09-02,RG_034,103.230001
2026-09-02,RG_041,101.549998
...,...,...
2026-09-03,RG_016,0.000000
2026-09-03,RG_017,0.000000
2026-09-03,RG_018,0.000000


In [135]:
# test out heavy rainfall day
mapped_polygons = thiessen.copy()
mapped_polygons= mapped_polygons.merge(
    all_precipitation.iloc[all_precipitation.index == all_precipitation.index[2]][["name", "rainfall"]], 
    left_on="gauge", right_on="name", how="left").drop(columns=["name"]).fillna({"rainfall": 0})  # Fill NaN values with 0 for gauges without rainfall data on that day
display(mapped_polygons.head())

delays = weather_bus_delays_gdf[weather_bus_delays_gdf["Date"].dt.normalize() == all_precipitation.index[2]]
delays = (
    delays[["Station", "Min Delay", "geometry"]]
    .groupby("Station", as_index=False)
    .agg({"Min Delay": "sum", "geometry": "first"})
)
display(delays.head())

,gauge,geometry,rainfall
0,RG_001,"POLYGON ((-79.47535 43.66324, -79.45558 43.663...",41.817
1,RG_003,"POLYGON ((-79.41549 43.64079, -79.42491 43.669...",53.673
2,RG_004,"POLYGON ((-79.42491 43.66966, -79.42881 43.673...",60.945
3,RG_006,"POLYGON ((-79.38886 43.65094, -79.39122 43.666...",59.957
4,RG_007,"POLYGON ((-79.35725 43.68072, -79.3181 43.6987...",60.720


,Station,Min Delay,geometry
0,1179 FINCH AVE W,20.0,POINT (-79.48311 43.76513)
1,BATHURST AND COLLEGE,24.0,POINT (-79.40764 43.65645)
2,CASTLEFIELD AND AVENUE,15.0,POINT (-79.41078 43.70946)
3,CASTLEFIELD AND DANESB,408.0,POINT (-79.45953 43.6988)
4,FINCH AND CHESWOOD,10.0,POINT (-79.47702 43.76646)


In [142]:
folium_map = folium.Map(
    location=[43.7, -79.4],
    zoom_start=10,
    tiles="OpenStreetMap",
    attr="© OpenStreetMap contributors",
)

folium.map.CustomPane("polygons", z_index=200).add_to(folium_map)
folium.map.CustomPane("markers", z_index=650).add_to(folium_map)

color_max = float(mapped_polygons["rainfall"].max()) if pd.notna(mapped_polygons["rainfall"].max()) and mapped_polygons["rainfall"].max() > 0 else 1.0
color_scale = linear.Blues_09.scale(0, color_max)

delay_values = pd.to_numeric(delays["Min Delay"], errors="coerce").dropna()
delay_min = float(delay_values.min()) if not delay_values.empty else 0.0
delay_max = float(delay_values.max()) if not delay_values.empty else 1.0
if delay_min == delay_max:
    delay_max = delay_min + 1.0
delay_color_scale = linear.OrRd_09.scale(0, 1)
delay_range = delay_max - delay_min

color_scale.caption = "Daily rainfall (mm)"
delay_color_scale.caption = (
    f"Minimum delay intensity (actual range: {delay_min:g}-{delay_max:g} minutes)"
)


folium.GeoJson(
    mapped_polygons,
    name="Thiessen rainfall",
    pane="polygons",
    style_function=lambda feature: {
        "fillColor": color_scale(feature["properties"]["rainfall"])
        if feature["properties"]["rainfall"] is not None else "#bdbdbd",
        "color": "#444444",
        "weight": 1,
        "fillOpacity": 0.5,
    },
    tooltip=GeoJsonTooltip(
        fields=["gauge", "rainfall"],
        aliases=["Gauge", "Daily rainfall total (mm)"],
        localize=True,
    ),
).add_to(folium_map)

for _, row in delays.iterrows():
    folium.CircleMarker(
        location=[row["geometry"].y, row["geometry"].x],
        radius=5,
        pane="markers",
        color="black",
        fill_color=delay_color_scale(
            0.3 + 0.7 * ((float(row["Min Delay"]) - delay_min) / delay_range)),
        fill=True,
        fill_opacity=1,
        stroke=True,
        weight=1,
        tooltip=folium.Tooltip(
            f"<b>Station:</b> {row['Station']}<br>"
            f"<b>Min Delay:</b> {row['Min Delay']} minutes"
        ),

    ).add_to(folium_map)


color_scale.add_to(folium_map)
delay_color_scale.add_to(folium_map)

folium.LayerControl().add_to(folium_map)

folium_map.save("thiessen_polygons_mapped.html")
folium_map

# SciKit Learn Intergration

In [ ]:
delays = bus_delays_data[weather_bus_delay_gp["Date"].dt.year >= 2025]
delays["Date"] = delays["Date"].dt.normalize()
delays = (
    delays[["date", "Station", "Min Delay", "geometry"]]
    .groupby("Station", "Date", as_index=False)
    .agg({"Min Delay": "sum", "geometry": "first"})
)
delays

C:\Users\nikit\AppData\Local\Temp\ipykernel_7268\2756649136.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  delays["Date"] = delays["Date"].dt.normalize()


KeyError: "['date', 'geometry'] not in index"

In [ ]:
from sklearn.linear_model import LinearRegression

# 638 dates with weather delays 2025 - 2026
delays  = bus_delays_data[bus_delays_data["Date"].dt.year >= 2025]
delays["Date"] = delays["Date"].dt.normalize().dropna().unique()
delays = (
    delays[["date","Station", "Min Delay", "geometry"]]
    .groupby("Station","Date", as_index=False)
    .agg({"Min Delay": "sum", "geometry": "first"})
)

train_dates = dates[:int(len(dates) * 0.8)]
test_dates = dates[int(len(dates) * 0.8):]
